# 02 — SentencePiece Tokenizer Training

## Why subword tokenization?
Urdu has rich morphology — word-level tokenization leads to huge vocabularies
and many out-of-vocabulary tokens. Subword methods split words into meaningful
pieces, balancing vocabulary size and coverage.

## Why SentencePiece?
SentencePiece operates directly on raw text without needing a pre-tokenizer,
which is ideal for Urdu (no clear word boundaries in some scripts).

## Why unigram model?
The unigram model learns a probabilistic vocabulary that maximises the
likelihood of the training corpus. It often gives better coverage for
morphologically rich languages compared to BPE.

## Why 8000 vocabulary?
8000 is the size specified in the assignment. It is a reasonable trade-off:
large enough to capture common subwords, small enough to train efficiently.

## Why character_coverage = 1.0?
We want every character in the Urdu text to be representable. Setting
coverage to 1.0 ensures no characters are dropped as unknown.

## Why `<ans>` and `</ans>` as user-defined symbols?
These markers delimit the answer span in the source sentence. They must
be kept as single tokens so the model can distinguish the answer region.

## Data leakage prevention
The tokenizer is trained ONLY on training data (sources + targets).
Validation and Wiki-UQA data are never included.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs.config import SP_MODEL_PATH, TRAIN_FILE

## Step 1: Train the SentencePiece tokenizer

In [ ]:
from src.tokenizer.train_tokenizer import train_sentencepiece

train_sentencepiece()

## Step 2: Load and inspect the trained tokenizer

In [ ]:
from src.tokenizer.tokenizer_utils import UrduTokenizer

tokenizer = UrduTokenizer(SP_MODEL_PATH)
print(f'Vocabulary size: {tokenizer.vocab_size}')
print(f'PAD={tokenizer.pad_id}  UNK={tokenizer.unk_id}  '
      f'BOS={tokenizer.bos_id}  EOS={tokenizer.eos_id}')

## Step 3: Verify answer markers are single tokens

In [ ]:
ans_open = tokenizer.piece_to_id('<ans>')
ans_close = tokenizer.piece_to_id('</ans>')
print(f'<ans>  → ID {ans_open}  (is single token: {ans_open != tokenizer.unk_id})')
print(f'</ans> → ID {ans_close}  (is single token: {ans_close != tokenizer.unk_id})')
assert ans_open != tokenizer.unk_id, '<ans> was split — check user_defined_symbols'
assert ans_close != tokenizer.unk_id, '</ans> was split — check user_defined_symbols'

## Step 4: Show 5 tokenization examples

For each example we show: original text → subword pieces → token IDs
→ reconstructed text, and verify that the round-trip is exact.

In [ ]:
examples = [
    'دریائے سندھ کی لمبائی <ans> 3,180 کلومیٹر </ans> ہے۔',
    'اسلام آباد <ans> پاکستان </ans> کا دارالحکومت ہے۔',
    '<ans> علامہ اقبال </ans> نے شکوہ لکھی۔',
    'پاکستان کی آبادی <ans> 220 ملین </ans> سے زیادہ ہے۔',
    'قائداعظم <ans> محمد علی جناح </ans> پاکستان کے بانی تھے۔',
]

for i, text in enumerate(examples, 1):
    print(f'── Example {i} ──')
    tokenizer.show_example(text)

## Step 5: Round-trip reconstruction test

We encode then decode each example and verify the original text
is perfectly recovered. This is a critical correctness check.

In [ ]:
import csv

# Test round-trip on actual training examples
passed = 0
failed = 0
with open(TRAIN_FILE, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f, delimiter='\t')
    for i, row in enumerate(reader):
        if i >= 100:
            break
        for text in [row['source'], row['target']]:
            if tokenizer.verify_round_trip(text):
                passed += 1
            else:
                failed += 1
                if failed <= 3:
                    print(f'Round-trip FAILED: {text[:80]}...')

print(f'\nRound-trip test: {passed} passed, {failed} failed out of {passed+failed}')